# CSC 421 - Knowledge and Reasoning 3
#### **Inference in First-order logic**

### Instructor: Shengyao Lu

## Readings

CHAPTER 9 INFERENCE IN FIRST-ORDER LOGIC 

* Basic: Sections 9.1, 9.5 Summary
* Expected: 9.2, 9.3, 9.4
* Advanced: All the chapter including bibliographical and historical notes

## 7. Theorem Proving in FOL

First-order inference can be reduced to propositional inference by paying an enormous price. 

- **Universal Instantiation (UI):** we can infer any sentence obtained by **substituting** a ground term (a term without variables) for $\forall$.
$$\frac{\forall x \,\ P(x)}{\text{Subst}(\{x/c\}, P(x))}, c=\text{ground term in KB}$$
$$i.e., \forall x \,\ P(x) \rightarrow P(c)$$
- **Existential Instantiation:**
$$\frac{\exist x \,\ P(x)}{\text{Subst}(\{x/k\}, P(x))}, k=\text{constant that does not appear elsewhere in KB}$$
$$i.e., \exist x \,\ P(x) \rightarrow P(k)$$
Here, $k$ is a new constant symbol, called Skolem constant. 

### 7.1 Unification
$$ \underset{premise}{\underbrace{A\land B\land \dots}}\rightarrow Q$$
Can we find a substitution $\theta$ to $x$, so that each of the conjuncts of the premise of the implication identical to sentences already in the KB?
$$\text{Unify}(p,q)=\theta \text{ where Subst}(\theta,p)=\text{Subst}(\theta,q)$$
$$e.g., \text{Unify}(King(x), King(John))=\{x / John\}$$

<img src="images/fol-unify-ex.png" width="700px">

- **Standardizing apart:** Eliminate conflicts on variable naming, e.g., $Knows(z_{17},OJ)$
- **Most general unifier (MGU):** For every unifiable pair, unique up to variable renaming. 
    - If you can remove part of a substitution and the expressions still unify, then the substitution is not an MGU.

#### ❓Q: $\mathbf{Unify(Knows(John,x), Knows(y,z))?\ \ MGU=?}$

Solution: $\theta=?$

#### **Psuedocode**
<img src="images/fol-unify-psuedo.png" width="700px">

(see demo)


### 7.2 Forward chaining

First-order definite clauses: disjunctions of literals of which exactly one is positive.
- e.g., $King(x)\land Greedy(x)\rightarrow Evil(x)$ 
- or $\neg King(x)\lor \neg Greedy(x)\lor Evil(x)$

<img src="images/fol-fc-psuedo.png" width="700px">

(see demo)

- **Pros**:
    - sound, complete
- **Cons**:
    - Not goal-directed
    - Poor efficiency when KB scales up
        1. inner loop tries to match every rule against every fact in KB
        2. rechecks every rule on every iteration
        3. Generate many irrelevant facts



#### **Addressing the Cons**
1. Missing rules against known facts
    - use heuristics to order the rules in the inner loop, e.g., MRV(minimum-remaining values)
2. Incremental forward chaining
    - recheck only rules involving new facts
3. Irrelevant facts
    - ignore it

(see demo)

### 7.3 Backward Chaining

Forward chaining is not "goal-directed". Backward chaining algorithms work backward from the goal, chaining through rules to find known facts that support the proof. 

<img src="images/fol-bc-psuedo.png" width="700px">

a DFS algorithm that can be **incomplete**
- __*FOL-BC-Ask*__: generator.
- __*FOL-BC-Or*__: fetch all clauses that might unify with the goal.
- __*FOL-BC-And*__: prove every conjunct in premise.

(see demo)

#### Risk of infinite loops with poor ordering
When there exists recursive rules:
> - $p(X) :- q(X).$
> - $q(X) :- p(X).$

It is possible:
> _prove p → prove q → prove p → prove q →_ $\dots$

<img src="images/fol-bc-infinite-loop.png" width="800px">

Can be alleviated by **memoization**.
- memorize what is being proved. Don't do that again
- memorize what has been proved True or False. Don't do that again. 

| Aspect                     | Forward Chaining (FC)                            | Backward Chaining (BC)                                 |
| -------------------------- | ------------------------------------------------ | ------------------------------------------------------ |
| **Inference direction**    | **facts → conclusions**                     | **query → required facts**                        |
||data-driven| goal-driven|
| **Typical implementation** | Datalog                      | Prolog                                                 |
| **Efficiency sweet spot**  | When many queries will be asked over the same KB | When only **a few specific queries** matter            |
| **Main strength**          | Systematically derives **all entailed facts**    | Focuses only on what is **relevant to the query**      |
| **Main weakness**          | Can generate many **irrelevant facts**           | Can suffer from **redundant search or infinite loops** |
| **Best used when**         | You want a **complete closure** of knowledge     | You want to **answer a specific question** quickly     |
|**Examples**| object recognition, routine decisions | Where are my keys? How do I get into a PhD program? |

<img src="images/pl-chain.png" width="600px">

### 7.3 Resolution

|                              Resolution in PL                              |                                                 Resolution in FOL                                                 |
|:--------------------------------------------------------------------------:|:-----------------------------------------------------------------------------------------------------------------:|
| remove complementary literals                                              | Unify first, then remove complementary literals                                                                   |
| $$\frac{\begin{matrix} P\lor Q \\ \neg P \lor R \end{matrix}}{Q \lor R} $$ | $$\frac{\begin{matrix} P(x)\lor Q(x) \\ \neg P(a) \lor R(y) \end{matrix}}{Q(a) \lor R(y)}, \text{when }\theta=\{x/a\}$$ |

$$
\frac{\ell_1 \vee \cdots \vee \ell_k,
\quad\quad
m_1 \vee \cdots \vee m_n}{
\mathrm{SUBST}\Big(
\theta,\,
\ell_1 \vee \cdots \vee \ell_{i-1} \vee \ell_{i+1} \vee \cdots \vee \ell_k
\vee
m_1 \vee \cdots \vee m_{j-1} \vee m_{j+1} \vee \cdots \vee m_n
\Big)}
$$

where
$$
\mathrm{UNIFY}(\ell_i,\,\neg m_j) = \theta.
$$

#### Example

We can resolve the two clauses
$$[Animal(F(x)) \,\vee\, Loves(G(x),x)]
\quad\text{and}\quad
[\neg Loves(u,v) \,\vee\, \neg Kills(u,v)]$$

by eliminating the complementary literals
$Loves(G(x),x)$ and $\neg Loves(u,v)$.

The unifier is
$$
\theta =  u/G(x),\, v/x .
$$

Applying this substitution produces the **resolvent clause**
$$
 Animal(F(x)) \,\vee\, \neg Kills(G(x),x) .
$$


#### **Example proof: West is the criminal**

To prove $KB\vDash \alpha$, we need to prove $KB\land\neg\alpha$ is unsatisfiable. 

(see demo)


## 8. Exercises

### Q: FOL + FC+ BC

<img src="images/fol-q1.png" width="950px">

### Q: PL + resolution

<img src="images/pl-q1.png" width="800px">